# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

### Challenge2 - Realizado por Jesús Carlos Sánchez Guzman sep 2026 ###

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Con Jupiter local - Ejecutar esta Celda Si se ejecuta de forma local con Jupyter, si es colab Ejecuitar la siguiente Celda
import os
from groq import Groq

# Antes de ejecutar, define la variable de entorno en tu sistema:
# En Linux/Mac:
# export GROQ_API_KEY="tu_api_key"
#
# En Windows (PowerShell):
# setx GROQ_API_KEY "tu_api_key"

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [ ]:
# NO EJECUTAR, si ya ejecutaste la anterior, SOLO PARA EJECUTARLO EN COLAB
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [2]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto.


In [20]:
# Prompt de clasificación en modo few-shot

# Prompt de clasificación en modo few-shot extendido

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento: Positivo

Reseña: "El corte de cabello fue impecable, justo lo que buscaba."
Sentimiento: Positivo

Reseña: "Nunca respetaron mi cita, tuve que esperar más de una hora."
Sentimiento: Negativo

Reseña: "El lugar estaba limpio, pero el servicio fue algo lento."
Sentimiento: Mixto

Reseña: "El barbero entendió perfectamente mi estilo, quedé muy satisfecho."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)


Few-shot: Reseña: "El barbero entendió perfectamente mi estilo, quedé muy satisfecho."  
Sentimiento: **Positivo**


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [21]:
# Razonamiento paso a paso (chain-of-thought) en contexto financiero

problema_finanzas = (
    "Juan tiene una tarjeta de crédito con un saldo de $5,000. "
    "El banco le cobra un interés mensual del 2%. "
    "Si Juan no paga nada durante 3 meses, ¿cuál será su deuda al final? "
    "Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot_finanzas = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema_finanzas}]
)

print(response_cot_finanzas.choices[0].message.content)


**Paso 1 – Identificar la tasa de interés y el método de acumulación**  
- Interés mensual: **2 %** (es decir, 0,02 en forma decimal).  
- El interés se aplica sobre el saldo que ya tiene el cliente, por lo que se **acumula mensualmente** (interés compuesto).

**Paso 2 – Calcular el saldo después del primer mes**  
Saldo inicial = **$5 000**  
Interés del primer mes = 5 000 × 0,02 = **$100**  
Saldo al final del primer mes = 5 000 + 100 = **$5 100**

**Paso 3 – Calcular el saldo después del segundo mes**  
Saldo al inicio del segundo mes = $5 100  
Interés del segundo mes = 5 100 × 0,02 = **$102**  
Saldo al final del segundo mes = 5 100 + 102 = **$5 202**

**Paso 4 – Calcular el saldo después del tercer mes**  
Saldo al inicio del tercer mes = $5 202  
Interés del tercer mes = 5 202 × 0,02 = **$104,04**  
Saldo al final del tercer mes = 5 202 + 104,04 = **$5 306,04**

---

### Resultado final
Después de **3 meses sin realizar pagos**, la deuda de Juan será:

\[
\boxed{\$5\,306,04}
\]


### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [23]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado del torneo interno de fútbol del TecNM campus Tuxtla Gutiérrez "
    "celebrado el 10 de septiembre de 2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Por más convincente que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación


No dispongo de datos sobre ese torneo.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [24]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [25]:
# Definir la base de conocimiento (reglamento escolar) y generar sus embeddings

from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Los estudiantes deben asistir al menos al 80% de las clases para poder presentar examen final.",
    "El uso de dispositivos móviles está permitido solo durante actividades académicas autorizadas.",
    "Las tareas entregadas después de la fecha límite tendrán una penalización del 20% en la calificación."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Modelo de embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Base de conocimiento: reglamento escolar
documentos = [
    "Los estudiantes deben asistir al menos al 80% de las clases para poder presentar examen final.",
    "El uso de dispositivos móviles está permitido solo durante actividades académicas autorizadas.",
    "Las tareas entregadas después de la fecha límite tendrán una penalización del 20% en la calificación."
]

# Generar embeddings de los documentos
embeddings_documentos = modelo_embeddings.encode(documentos)

# Función para buscar el fragmento más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

# Ejemplo de consulta en contexto escolar
pregunta = "¿Qué pasa si falto muchas clases?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fragmento recuperado: Los estudiantes deben asistir al menos al 80% de las clases para poder presentar examen final.


In [28]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del estudiante usando SOLO el siguiente reglamento escolar. 
Si el reglamento no cubre la pregunta, dilo claramente.

Reglamento: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una regla genérica


No podrás presentar el examen final.
